# International Football Results: Penalties, Home Advantage and Elo Ratings
Analysis of international football matches from 1872 onwards using Python, SQL and pandas.
Data source: [Kaggle – International football results](https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017)


   **Questions:**
   1. Which national teams have been the strongest in each historical era (Elo rating)?
   2. Do teams score more goals at home than away?
   3. Do home teams score more penalties, and does this explain the home advantage?

In [34]:
import pandas as pd
import numpy as np
import sqlite3 as sql
from collections import defaultdict

## 1. Data Loading
Loading the four CSV files from GitHub into an in-memory SQLite database.

In [4]:
base = 'https://raw.githubusercontent.com/m1taxx/international-football-analysis/48646b450f8778c3ac74ca263ee3b25a51734759/Football%20data/'

con = sql.connect(':memory:')

for name in ['former_names', 'goalscorers', 'results', 'shootouts']:
    pd.read_csv(base + name + '.csv').to_sql(name, con, index=False, if_exists='replace')


df_former_names = pd.read_sql( 'SELECT * FROM former_names' , con)
df_goalscorers = pd.read_sql( 'SELECT * FROM goalscorers', con)
df_shootouts = pd.read_sql('SELECT * FROM shootouts', con)
df_results = pd.read_sql('SELECT * FROM results', con)


## 2. Elo Ratings
**Question:** Which national teams have been the strongest over the history of international football?

### 2.1 Data Preparation and Tournament Weights (K-factor)
More important tournaments get a higher K-factor: major finals – 60, qualifiers and Nations League – 40, other matches – 30.

In [5]:


df_filtred_results = df_results[df_results['tournament'] != 'Friendly'].sort_values(by = 'date', ascending=True )

def assign_k_factor(tournament_name):
  name = str(tournament_name)
  if any( x in name for x in['World Cup', 'UEFA Euro', 'Copa América', 'African Cup of Nations',
                               'AFC Asian Cup', 'Gold Cup', 'Confederations Cup', 'Oceania Nations Cup']) \
     and 'qualification' not in name.lower():
      return 60
  elif 'qualification' in name.lower() or 'Nations League' in name or 'Championship' in name :
    return 40
  else:
    return 30
df_filtred_results['k_factor'] = df_filtred_results['tournament'].apply(assign_k_factor)


### 2.2 Elo Calculation
All teams start at 1000 points. Ratings are updated after every match in chronological order.

Epochs
Elo ratings are calculated separately for each historical era, and all ratings are reset to 1000 at the start of each one. This shows which teams were strongest *within* a given period, rather than carrying over the advantage of older results.

The boundaries are based on major changes in international football:

| Era | Years | Why |
|---|---|---|
| 1 | 1872–1945 | Early period: the first World Cup (1930), few matches, mostly Europe and South America. Ends with World War II. |
| 2 | 1946–1969 | Post-war recovery: the World Cup resumes in 1950, many new FIFA members after decolonisation. |
| 3 | 1970–1989 | Cold War era: regular qualification tournaments, more matches for African and Asian teams. |
| 4 | 1990–2009 | Dissolution of the USSR, Yugoslavia and Czechoslovakia creates dozens of new national teams; the World Cup expands to 32 teams in 1998. |
| 5 | 2010–now | Modern era, including the UEFA Nations League (since 2018). |

**Note:** the boundaries are approximate and chosen for readability, not as strict historical cut-offs. Only teams with at least 20 matches in an era are shown, to avoid small-sample noise.

In [6]:
epoch_bounds = {
    '1872–1945': ('1872-01-01', '1945-12-31'),
    '1946–1969': ('1946-01-01', '1969-12-31'),
    '1970–1989': ('1970-01-01', '1989-12-31'),
    '1990–2009': ('1990-01-01', '2009-12-31'),
    '2010–now':  ('2010-01-01', '2099-12-31'),
}

In [7]:
def calculate_elo(matches, start_rating=1000.0):
    ratings = defaultdict(lambda: start_rating)
    games = defaultdict(int)

    for row in matches.itertuples():
        home_rating = ratings[row.home_team]
        away_rating = ratings[row.away_team]

        if row.home_score > row.away_score:
            home_result, away_result = 1.0, 0.0
        elif row.home_score < row.away_score:
            home_result, away_result = 0.0, 1.0
        else:
            home_result, away_result = 0.5, 0.5

        home_expected = 1 / (1 + 10 ** ((away_rating - home_rating) / 400))
        away_expected = 1 - home_expected

        ratings[row.home_team] = home_rating + row.k_factor * (home_result - home_expected)
        ratings[row.away_team] = away_rating + row.k_factor * (away_result - away_expected)

        games[row.home_team] += 1
        games[row.away_team] += 1

    return (
        pd.DataFrame({
            'team': list(ratings),
            'points': [round(ratings[t]) for t in ratings],
            'matches': [games[t] for t in ratings],
        })
        .sort_values('points', ascending=False)
        .reset_index(drop=True)
    )





### 2.3 Final Rating Table
Top 20 teams by Elo rating.

In [8]:
MIN_MATCHES = 20
epoch_ratings = {}

for era, (start, end) in epoch_bounds.items():
    matches = df_filtred_results[
        (df_filtred_results['date'] >= start) & (df_filtred_results['date'] <= end)
    ]
    table = calculate_elo(matches)
    table = table[table['matches'] >= MIN_MATCHES].head(20).reset_index(drop=True)
    table.index += 1
    epoch_ratings[era] = table

    print(f'Top 20 teams: {era}')
    display(table)

Top 20 teams: 1872–1945


,team,points,matches
1,Italy,1289,56
2,Argentina,1278,178
3,Brazil,1220,71
4,China,1182,22
5,Uruguay,1172,152
6,Hungary,1167,44
7,Guernsey,1164,58
8,Costa Rica,1098,25
9,Czechoslovakia,1091,52
10,Sweden,1086,63


Top 20 teams: 1946–1969


,team,points,matches
1,England,1300,106
2,Germany,1270,44
3,Brazil,1233,148
4,Uruguay,1220,122
5,Costa Rica,1198,89
6,Italy,1190,62
7,Russia,1189,54
8,Argentina,1170,117
9,DR Congo,1157,25
10,Czechoslovakia,1156,75


Top 20 teams: 1970–1989


,team,points,matches
1,Brazil,1328,100
2,Netherlands,1304,100
3,South Korea,1303,277
4,Cameroon,1258,104
5,Germany,1252,104
6,Tahiti,1229,40
7,Italy,1214,89
8,Mexico,1207,51
9,Russia,1207,101
10,Spain,1195,81


Top 20 teams: 1990–2009


,team,points,matches
1,Spain,1446,142
2,Brazil,1423,190
3,Netherlands,1373,131
4,Germany,1369,139
5,Mexico,1362,222
6,Italy,1342,151
7,United States,1331,191
8,England,1312,129
9,Australia,1307,110
10,Cameroon,1306,156


Top 20 teams: 2010–now


,team,points,matches
1,Spain,1613,156
2,Argentina,1492,144
3,England,1477,149
4,France,1461,145
5,Mexico,1424,155
6,Morocco,1407,126
7,Portugal,1398,153
8,Netherlands,1349,143
9,Colombia,1333,116
10,Japan,1324,166


**Conclusion:** **Italy** leads the 1872–1945 ranking, **England** the 1946–1969 era, **Brazil** 1970–1989, and **Spain** both 1990–2009 and 2010–now. **Brazil** is in the top three in each of the first four eras but only 12th in 2010–now, while **Germany** drops to 18th in the same period. Some results are artifacts of the method rather than football reality: Jersey appears in the top 20 in three eras and Guernsey ranks 7th in 1872–1945, and **Tahiti** ranks 6th in 1970–1989, probably because these teams play mostly against weak opponents and are rarely tested against strong ones. This is a known limitation of Elo in a poorly connected network of teams. Points are also not comparable across eras: the spread grows over time partly because teams play more matches.

## 3. Home Advantage
**Question:** Do teams score more goals at home than away, and does this change when friendlies are excluded?

### 3.1 Home vs Away Goals (All Matches)

In [35]:
query = '''
    WITH home_stats AS (
        SELECT home_team AS team, SUM(home_score) AS home_goals
        FROM results
        WHERE neutral = 0
        GROUP BY home_team
    ),
    away_stats AS (
        SELECT away_team AS team, SUM(away_score) AS away_goals
        FROM results
        WHERE neutral = 0
        GROUP BY away_team
    )
    SELECT
        team,
        home_goals,
        away_goals,
        home_goals + away_goals AS total_goals,
        home_goals * 1.0 / away_goals AS goal_ratio
    FROM home_stats h
    JOIN away_stats USING(team)
    WHERE total_goals >= 500
    ORDER BY goal_ratio DESC

'''

home_ratio = pd.read_sql(query, con)






In [36]:
home_ratio.head(5)


,team,home_goals,away_goals,total_goals,goal_ratio
0,Malaysia,710,186,896,3.817204
1,Saudi Arabia,565,153,718,3.692810
2,Qatar,493,135,628,3.651852
3,United States,855,256,1111,3.339844
4,Singapore,461,141,602,3.269504


   **Conclusion:** With friendlies included, the top of the list is dominated by Asian teams (Malaysia, Saudi Arabia, Qatar, Singapore) and the United States, with a home/away goal ratio of 3.3 to 3.8. This ratio is based on total goals, not goals per match, so it can be distorted by teams that play many more home than away matches. Section 3.2 corrects for this and also excludes friendlies.

### 3.2 Home vs Away Goals (Excluding Friendlies)

In [11]:
query_non_friendly = '''
WITH home_stats AS (
    SELECT
        home_team AS team,
        SUM(home_score) AS home_goals,
        COUNT(*) AS home_matches
    FROM results
    WHERE tournament != 'Friendly' AND neutral = 0
    GROUP BY home_team
),
away_stats AS (
    SELECT
        away_team AS team,
        SUM(away_score) AS away_goals,
        COUNT(*) AS away_matches
    FROM results
    WHERE tournament != 'Friendly' AND neutral = 0
    GROUP BY away_team
)
SELECT
    team,
    home_goals,
    away_goals,
    home_goals + away_goals AS total_goals,
    home_matches,
    away_matches,
    home_matches + away_matches AS total_matches,
    home_goals * 1.0 / away_goals AS home_goal_ratio,
    home_goals * 1.0 / home_matches AS home_gpm,
    away_goals * 1.0 / away_matches AS away_gpm,
    (home_goals * 1.0 / home_matches) / (away_goals * 1.0 / away_matches) AS gpm_ratio
FROM home_stats
JOIN away_stats USING(team)
WHERE home_matches >= 50 AND away_matches >= 50
ORDER BY gpm_ratio DESC
'''
home_non_friendly_ratio = pd.read_sql(query_non_friendly, con)




In [12]:
home_non_friendly_ratio


,team,home_goals,away_goals,total_goals,home_matches,away_matches,total_matches,home_goal_ratio,home_gpm,away_gpm,gpm_ratio
0,Niger,71,27,98,50,67,117,2.629630,1.420000,0.402985,3.523704
1,Libya,120,47,167,70,85,155,2.553191,1.714286,0.552941,3.100304
2,Bolivia,212,91,303,122,133,255,2.329670,1.737705,0.684211,2.539723
3,Oman,168,75,243,84,94,178,2.240000,2.000000,0.797872,2.506667
4,Uganda,232,83,315,132,118,250,2.795181,1.757576,0.703390,2.498722
...,...,...,...,...,...,...,...,...,...,...,...
150,Bosnia and Herzegovina,143,129,272,89,93,182,1.108527,1.606742,1.387097,1.158349
151,Georgia,108,92,200,91,89,180,1.173913,1.186813,1.033708,1.148113
152,Latvia,141,130,271,129,127,256,1.084615,1.093023,1.023622,1.067800
153,Ukraine,116,136,252,77,95,172,0.852941,1.506494,1.431579,1.052330


In [13]:
df = home_non_friendly_ratio

df['gpm_diff'] = df['home_gpm'] - df['away_gpm']

df['gpm_ratio'].median(), df['gpm_diff'].median(), (df['gpm_ratio'] < 1).sum()

(1.6842105263157894, 0.7244623655913978, np.int64(1))

**Conclusion (home advantage, excluding friendlies and neutral-venue matches):** Among 155 teams with at least 50 home and 50 away matches, almost all score more goals per match at home than away: only Barbados (0.97) has a ratio below 1. The median ratio is 1.68, meaning a typical team scores about 70% more goals per match at home, or roughly 0.72 goals more per match. The strongest home effect is seen for Niger (3.52), Libya (3.10), Bolivia (2.54), Oman (2.51) and Uganda (2.50). In several of these cases the ratio is driven by very low away scoring (e.g. Niger scores 0.40 goals per match away vs 1.42 at home), and Niger has exactly 50 home matches, so its value is based on a small sample. The smallest differences are for Bosnia and Herzegovina, Georgia, Latvia and Ukraine, where home and away scoring are almost equal; Ukraine scores 1.51 goals per match at home and 1.43 away. This may reflect shorter travel distances in Europe or stronger away performance, but these explanations were not tested here.

## 4. Penalties
**Question:** Do home teams score more penalties per match than away teams, and does this explain the home advantage found in section 3?

In section 3 we saw that a typical team scores about 70% more goals per match at home than away. One possible contributor is penalties. The `goalscorers` table only records goals that were scored, so we cannot say how many penalties were awarded; we can only compare scored penalties at home and away, both per match and as a share of all goals.

**Data notes:**
- Penalty records in the dataset start in 1916, so earlier matches are excluded from this section.
- Own goals are excluded from goal counts.
- `goalscorers` covers only about 32% of goals recorded in `results` since 1916, so absolute penalty counts are underestimated. The comparison is made within each team (home vs away) and as a share of recorded goals.
- Only matches with at least one recorded goal are used. Matches without any recorded goal (e.g. 0–0) are excluded, while matches with only partially recorded goals may still be included.
- Only teams in the top 25% by number of such matches (more than 216 home and away matches combined) are included, to avoid small-sample noise: 55 teams.

### 4.1 Data Coverage Check
How many of the goals recorded in `results` also appear in `goalscorers` since 1916? This shows how reliable the penalty counts are.

In [14]:
date_period = '1916-07-02'
coverage = '''
SELECT
    (SELECT COUNT(*) FROM goalscorers WHERE date >= :d AND own_goal = 0) AS goals_in_goalscorers,
    (SELECT SUM(home_score + away_score) FROM results WHERE date >= :d) AS goals_in_results
'''
pd.read_sql(coverage, con, params={'d': date_period})


,goals_in_goalscorers,goals_in_results
0,46978,144630


### 4.2 Penalties per Match, Home vs Away
For each team: scored penalties and goals at home and away, number of matches with recorded goals, penalties per match, and the difference between home and away.

In [32]:
query_penalties = '''
WITH home_stats AS (
    SELECT home_team AS team,
           SUM(CASE WHEN team = home_team AND penalty = 1 THEN 1 ELSE 0 END) AS home_penalty,
           SUM(CASE WHEN team = home_team AND own_goal = 0 THEN 1 ELSE 0 END) AS home_goals
    FROM goalscorers AS g
    WHERE date >= :date_period
    GROUP BY home_team
),
away_stats AS (
SELECT away_team AS team,
       SUM(CASE WHEN away_team = team AND penalty = 1 THEN 1 ELSE 0 END) AS away_penalty,
       SUM(CASE WHEN away_team = team  AND own_goal = 0 THEN 1 ELSE 0 END) AS away_goals
FROM goalscorers  AS g
WHERE date >= :date_period
GROUP BY away_team
),
home_matches AS (
SELECT home_team AS team,
       COUNT(*) AS home_matches
FROM results AS r
WHERE date >= :date_period
    AND EXISTS (
          SELECT 1
          FROM goalscorers g
          WHERE g.date = r.date
            AND g.home_team = r.home_team
            AND g.away_team = r.away_team
      )
GROUP BY home_team
),
away_matches AS (
SELECT away_team AS team,
       COUNT(*) AS away_matches
FROM results AS r
WHERE date >= :date_period
    AND EXISTS (
          SELECT 1
          FROM goalscorers g
          WHERE g.date = r.date
            AND g.home_team = r.home_team
            AND g.away_team = r.away_team
      )
GROUP BY away_team

)
SELECT team,
  home_penalty,
  home_goals,
  away_penalty,
  away_goals,
  home_matches,
  away_matches,
  home_penalty * 1.0 / home_matches AS penalty_pm_home,
  away_penalty * 1.0 / away_matches AS penalty_pm_away,
  (home_penalty * 1.0 / home_matches) - (away_penalty * 1.0 / away_matches) AS penalty_dif
FROM home_stats
JOIN away_stats USING(team)
JOIN home_matches USING(team)
JOIN away_matches USING(team)
ORDER BY penalty_dif DESC

'''
con.execute('CREATE INDEX IF NOT EXISTS idx_gs_match ON goalscorers(date, home_team, away_team)')
df_penalties = pd.read_sql(query_penalties,con,params={'date_period': date_period})


### 4.3 Selecting Teams
Only teams in the top 25% by number of matches with recorded goals are kept, to avoid small-sample noise.

In [37]:
total_matches = df_penalties['home_matches'] + df_penalties['away_matches']
matches_quantile = total_matches.quantile(0.75)
df_penalties_filtred = df_penalties[total_matches > matches_quantile].reset_index(drop=True).copy()

### 4.4 Overall Results
Total penalties per match and share of goals from penalties at home and away, plus the number of teams with more penalties at home.

In [31]:
df = df_penalties_filtred

pen_pm_home = df['home_penalty'].sum() / df['home_matches'].sum()
pen_pm_away = df['away_penalty'].sum() / df['away_matches'].sum()

share_home = df['home_penalty'].sum() / df['home_goals'].sum() * 100
share_away = df['away_penalty'].sum() / df['away_goals'].sum() * 100

n_higher_home = (df['penalty_dif'] > 0).sum()
median_diff = df['penalty_dif'].median()

len(df), matches_quantile, pen_pm_home, pen_pm_away, share_home, share_away, n_higher_home, median_diff

(55,
 np.float64(216.5),
 np.float64(0.14006849315068493),
 np.float64(0.08927926843941764),
 np.float64(7.066344160331721),
 np.float64(6.6481498073649306),
 np.int64(48),
 0.04602410786602505)

### 4.5 Teams with More Penalties Away
The few teams where the home–away difference is negative.

In [38]:
df_penalties_filtred.head(10)

,team,home_penalty,home_goals,away_penalty,away_goals,home_matches,away_matches,penalty_pm_home,penalty_pm_away,penalty_dif
0,Austria,36,354,12,203,161,151,0.223602,0.079470,0.144132
1,Russia,29,356,9,269,156,168,0.185897,0.053571,0.132326
2,Bulgaria,30,282,9,154,159,149,0.188679,0.060403,0.128277
3,Australia,31,468,11,169,137,108,0.226277,0.101852,0.124426
4,Costa Rica,20,323,4,183,146,141,0.136986,0.028369,0.108618
5,Denmark,34,338,18,256,163,169,0.208589,0.106509,0.102080
6,Spain,40,557,20,371,204,205,0.196078,0.097561,0.098517
7,Republic of Ireland,26,251,14,191,148,157,0.175676,0.089172,0.086504
8,Switzerland,25,324,14,234,155,172,0.161290,0.081395,0.079895
9,Cyprus,16,138,6,80,134,137,0.119403,0.043796,0.075607


**Conclusion:** Among the 55 teams in the top 25% by number of matches with recorded goals since 1916, 48 score more penalties per match at home than away. Together they average 0.140 penalties per home match and 0.089 per away match (about 1.6 times more at home), and the median gap is 0.046 penalties per match, roughly one extra penalty for every 22 home games. The largest gaps are for Austria (36 penalties at home vs 12 away), Russia (29 vs 9) and Bulgaria (30 vs 9), followed by Australia, Costa Rica and Denmark. The seven exceptions, where teams score more penalties away, are Honduras, Japan, Romania, South Korea, Nigeria, Iran and Israel, and the differences there are small (at most 0.044 per match); five of them are non-European teams, which may reflect different refereeing or match conditions, but this was not tested here.

Overall, penalties make up a very similar share of goals at home and away (7.07% vs 6.65%). This suggests that home teams score more penalties mainly because they score more goals overall, so penalties explain only a small part of the home advantage found in section 3. For individual teams, however, the share can differ noticeably: Austria scores about 10.2% of its home goals from penalties vs 5.9% away, and Costa Rica 6.2% vs 2.2%, so for some teams penalties may matter more.

The sample is limited to teams with long histories, and `goalscorers` covers only about a third of all goals, so absolute values are underestimated; the home–away comparison within each team is more reliable than the totals. We also observe only scored penalties, so this cannot show whether home teams are awarded more of them.

## 5. Summary

**Key findings**
- **Elo by era:** Italy leads 1872–1945, England 1946–1969, Brazil 1970–1989 and Spain both 1990–2009 and 2010–now. Brazil is in the top three in each of the first four eras but only 12th in 2010–now.
- **Home advantage:** among 155 teams with at least 50 home and 50 away matches (friendlies and neutral-venue matches excluded), only Barbados scores fewer goals per match at home than away. The median home/away ratio is 1.68.
- **Penalties:** among 55 teams with long histories (since 1916), 48 score more penalties per match at home (0.140 vs 0.089), but penalties make up almost the same share of goals at home and away (7.07% vs 6.65%). They explain very little of the home advantage.

**Limitations**
- Elo does not model home advantage or goal difference, and renamed teams (e.g. USSR and Russia) are not merged. Teams that play mostly weak opponents, such as Jersey, can rank too high. Matches decided by penalty shootouts are counted as draws.
- `goalscorers` covers only about 32% of goals recorded in `results`, so penalty counts are underestimated; only the home–away comparison within each team is reliable. Only scored penalties are recorded, so awarded penalties cannot be analysed.
- The home advantage and penalty tables describe teams with long histories and do not represent smaller national teams.

**Next steps**
- Merge renamed teams using `former_names`.
- Test how well Elo predicts match results.
- Add visualisations in Plotly or Power BI.